In [1]:
import pandas as pd

In [2]:
people = pd.read_csv("../data/clean/df_people.csv", dtype={
    "height": "Int64", "mass": "Int64", "id_homeworld": "Int64", "id_species": "Int64"})
planets = pd.read_csv("../data/clean/df_planets.csv", dtype={ 
    "rotation_period": "Int64", "orbital_period": "Int64", "diameter": "Int64", "population": "Int64", "id_people": "Int64"})
species = pd.read_csv("../data/clean/df_species.csv", dtype={
    "average_height": "Int64", "id_homeworld": "Int64", "id_people": "Int64"})

In [3]:
people["id"] = range(1, len(people) + 1)
planets["id"] = range(1, len(planets) + 1)
species["id"] = range(1, len(species) + 1)

In [4]:
import sqlite3

In [5]:
conn = sqlite3.connect("starwars.db")

In [6]:
people.to_sql("people", conn, if_exists="replace", index=False) # index=False because I created it
planets.to_sql("planets", conn, if_exists="replace", index=False)
species.to_sql("species", conn, if_exists="replace", index=False)

37

In [7]:
# Character with their species and planet
query1 = """
SELECT 
    p.name as character_name,
    s.name as species,
    pl.name as planet_name
FROM people p
LEFT JOIN species s ON s.id = p.id_species 
LEFT JOIN planets pl ON pl.id = p.id_homeworld
"""

In [8]:
result1 = pd.read_sql_query(query1, conn)
print(result1)

     character_name  species planet_name
0    Luke Skywalker      NaN    Tatooine
1             C-3PO    Droid    Tatooine
2             R2-D2    Droid       Naboo
3       Darth Vader      NaN    Tatooine
4       Leia Organa      NaN    Alderaan
..              ...      ...         ...
77         Grievous  Kaleesh       Kalee
78          Tarfful   Wookie    Kashyyyk
79  Raymus Antilles      NaN    Alderaan
80        Sly Moore      NaN      Umbara
81       Tion Medon   Pau'an      Utapau

[82 rows x 3 columns]


In [9]:
# Percentage of human character for each planet
query2 = """
SELECT 
    pl.name as planet_name,
    CAST(SUM(CASE WHEN s.name = 'Human' THEN 1 ELSE 0 END) * 100.0 / count(p.id) AS TEXT) || '%' as per_human_by_planet
FROM people p
JOIN species s ON s.id = p.id_species 
JOIN planets pl ON pl.id = p.id_homeworld 
GROUP BY pl.name
HAVING SUM(CASE WHEN s.name = 'Human' THEN 1 ELSE 0 END) * 100.0 / count(p.id) > 0.0
ORDER BY CAST(per_human_by_planet AS FLOAT) DESC
"""

In [10]:
result2 = pd.read_sql_query(query2, conn)
print(result2)

  planet_name per_human_by_planet
0     Serenno              100.0%
1    Alderaan              100.0%
2   Coruscant               50.0%
3       Naboo               20.0%


In [11]:
# Gender's distribution by planet and species!
query3 = """
SELECT
    pl.name as planet_name,
    coalesce(s.name, "Unknown") as species,
    sum(case when p.gender = 'male' then 1 else 0 end) as nb_male,
    sum(case when p.gender = 'female' then 1 else 0 end) as nb_female
from people p
left JOIN planets pl ON pl.id = p.id_homeworld 
left JOIN species s ON s.id = p.id_species 
GROUP BY pl.name, s.name
"""

In [12]:
result3 = pd.read_sql_query(query3, conn)
print(result3)

       planet_name         species  nb_male  nb_female
0              NaN         Unknown        2          1
1              NaN           Droid        0          0
2              NaN  Yoda's species        1          0
3         Alderaan         Unknown        1          1
4         Alderaan           Human        1          0
5      Aleen Minor          Aleena        1          0
6           Bespin         Unknown        1          0
7       Bestine IV         Unknown        1          0
8   Cato Neimoidia       Neimodian        1          0
9            Cerea          Cerean        1          0
10        Champala        Chagrian        1          0
11       Chandrila         Unknown        0          1
12    Concord Dawn         Unknown        1          0
13        Corellia         Unknown        2          0
14       Coruscant         Unknown        1          0
15       Coruscant           Human        0          1
16       Coruscant      Tholothian        0          1
17        

In [13]:
# Top 3 most diverse planet
query4 = """
SELECT 
    COALESCE(pl.name, "Unknown") as planet_name,
    count(distinct s.name) as nb_species
from people p
JOIN planets pl ON pl.id = p.id_homeworld 
JOIN species s ON s.id = p.id_species
group by pl.name
order by nb_species desc limit 3
"""

In [14]:
result4 = pd.read_sql_query(query4, conn)
print(result4)

  planet_name  nb_species
0       Naboo           3
1   Coruscant           2
2     Unknown           2


In [15]:
# Character, their species and average height by species
query5 = """
select 
    p.name as character_name,
    s.name as character_species,
    p.height as character_height,
    round(avg(p.height) over(partition by s.name),2) as avg_height_per_species
from people p
left JOIN species s ON s.id = p.id_species 
order by p.name
"""

In [16]:
result5 = pd.read_sql_query(query5, conn)
display(result5)

,character_name,character_species,character_height,avg_height_per_species
0,Ackbar,Mon Calamari,180.0,180.00
1,Adi Gallia,Tholothian,184.0,184.00
2,Anakin Skywalker,NaN,188.0,174.84
3,Arvel Crynyd,NaN,NaN,174.84
4,Ayla Secura,Twi'lek,178.0,179.00
...,...,...,...,...
77,Wicket Systri Warrick,Ewok,88.0,88.00
78,Wilhuff Tarkin,NaN,180.0,174.84
79,Yarael Poof,Quermian,264.0,264.00
80,Yoda,Yoda's species,66.0,66.00


In [17]:
# Rank of planet by speceis diversity
query6 = """
select 
    coalesce(pl.name, "Unknown") as planet_name,
    dense_rank() over(order by count(distinct s.name) desc) as most_species
from people p
LEFT JOIN planets pl ON pl.id = p.id_homeworld 
LEFT JOIN species s ON s.id = p.id_species 
group by pl.name
"""

In [18]:
result6 = pd.read_sql_query(query6, conn)
print(result6)

       planet_name  most_species
0            Naboo             1
1        Coruscant             2
2          Unknown             2
3            Zolan             3
4          Vulpter             3
5           Utapau             3
6             Tund             3
7          Troiken             3
8        Trandosha             3
9         Toydaria             3
10        Tatooine             3
11         Sullust             3
12           Skako             3
13           Shili             3
14         Serenno             3
15          Ryloth             3
16           Rodia             3
17         Quermia             3
18            Ojom             3
19       Nal Hutta             3
20      Muunilinst             3
21        Mon Cala             3
22          Mirial             3
23       Malastare             3
24        Kashyyyk             3
25          Kamino             3
26           Kalee             3
27        Iridonia             3
28         Iktotch             3
29     Gle